In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [27]:
# %%
import datetime
import logging
import os
import re

import pandas as pd
# /venv/lib/python3.12/site-packages/gspread_pandas/spread.py:401: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)` .replace("", np.nan)
pd.set_option('future.no_silent_downcasting', True)

import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hpandas as hpandas
import helpers.hprint as hprint
import helpers.hcache as hcache

#hcache.get_global_cache_info()
#hcache.clear_global_cache("all")

import config_root.config as cconfig

# %%
hdbg.init_logger(verbosity=logging.INFO)

_LOG = logging.getLogger(__name__)

_LOG.info("%s", henv.get_system_signature()[0])

hprint.config_notebook()

INFO  # Git
  branch_name='CmampTask11020_Compute_yamm_stats'
  hash='50f413699'
  # Last commits:
    *   50f413699 GP Saggese Merge branch 'master' into CmampTask11020_Compute_yamm_stats      (70 minutes ago) Wed Jan 8 23:11:29 2025  (HEAD -> CmampTask11020_Compute_yamm_stats, origin/CmampTask11020_Compute_yamm_stats)
    |\  
    * | 91b7ff63d GP Saggese Update                                                            (70 minutes ago) Wed Jan 8 23:11:23 2025           
    | * b682dff49 Dan      Cm task11040 remove get universe from im client 8 (#11164)        (   4 hours ago) Wed Jan 8 20:27:04 2025  (origin/master, origin/HEAD, master)
# Machine info
  system=Linux
  node name=a835e9cd7a0d
  release=6.6.22-linuxkit
  version=#1 SMP Fri Mar 29 12:21:27 UTC 2024
  machine=aarch64
  processor=aarch64
  cpu count=8
  cpu freq=None
  memory=svmem(total=8222072832, available=6460047360, percent=21.4, used=1552101376, free=2254422016, active=1914208256, inactive=3219775488, buffers=6761

In [3]:
import gspread
print(gspread.__version__)

import gspread_pandas
print(gspread_pandas.__version__)

#gspread_pandas.conf.get_config()
print(gspread_pandas.conf.get_config()["project_id"])

#!sudo /bin/f bash -c "(source /venv/bin/activate; pip install --upgrade google-api-python-client)"

import importlib
import ck_marketing.process_automation.hyamm as hyamm
importlib.reload(hyamm)

# import ck_marketing.hunterio.hunter_api as cmhuhuap
# importlib.reload(cmhuhuap)

#import helpers.hopenai as hopenai

5.12.4
3.3.0
gspread-gp


/app/ck_marketing/process_automation/hyamm.py:19: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


<module 'ck_marketing.process_automation.hyamm' from '/app/ck_marketing/process_automation/hyamm.py'>

# Read data

In [38]:
#url = "https://docs.google.com/spreadsheets/d/1nlfOHLUo2iTuNtFb2T5ZJXZZnztAnr3WzG9Jm5lbk6I"
#url = "https://docs.google.com/spreadsheets/d/1NCBbacXYEjnXtPBO1gn937lOUmum4Fvfw-fRxz5wvmY"
url = "https://docs.google.com/spreadsheets/d/1wuCds_rPTqjlu5mTUOKQC3pNBVXurTTZPdX5ryQr3wA"
contact_df = hyamm.get_cached_sheet_to_df(url, "Sheet3")
contact_df.set_index("hash", drop=True, inplace=True)
print(contact_df.shape)
display(contact_df.head(2))

INFO  Updating cached version on disk ...
INFO  Reading data from url='https://docs.google.com/spreadsheets/d/1wuCds_rPTqjlu5mTUOKQC3pNBVXurTTZPdX5ryQr3wA' sheet_name='Sheet3'
INFO  Updating cached version on disk done (2.314 s)
(607, 19)


,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,hunterio.email,hunterio.email_verification
hash,,,,,,,,,,,,,,,,,,,
4a4079215e5eccf89e6089f7dd6574f7,Folkapp,,David,Rossow,david.rossow@gatesfoundation.org,valid,_nan_,_nan_,,_nan_,http://www.gatesfoundation.org,New York,FALSE,,Health;Industry;Food;Consulting & Services;Sof...,Family Office,,david.rossow@gatesfoundation.org,valid
bf436a1cde88d1e0079a889081d32f88,Folkapp,,Jeff,Ehlers,jehlers@dnscap.com,valid,_nan_,_nan_,,_nan_,www.dnscap.com,Chicago;Illinois,FALSE,,Software & Internet;AI & Machine Learning;Logi...,Family Office,,jehlers@dnscap.com,valid


In [39]:
import ck_marketing.linkedin.linkedin_utils as cmliliut

In [40]:
df = contact_df.head(1000)

#df["first_name"].apply(lambda x : cmliliut.clean_first_last_name(x, last_name_only=False))
#df["last_name"].apply(lambda x : cmliliut.clean_first_last_name(x, last_name_only=True))

In [41]:
# Define a wrapper function to handle precedence of nicknames.
def process_names(row) -> pd.Series:
    first_name_cleaned, first_nickname = cmliliut.clean_first_last_name(
        row["first_name"], last_name_only=False
    )
    last_name_cleaned, last_nickname = cmliliut.clean_first_last_name(
        row["last_name"], last_name_only=True
    )
    m = re.match(r'^(\w+)\S+\s', first_name_cleaned)
    if m:
        first_name_cleaned = m.group(0)
    # Choose the nickname with precedence given to the first name.
    return pd.Series([first_name_cleaned, last_name_cleaned, first_nickname, last_nickname])


# Apply the processing function.
df_first_last = df.copy()
df_first_last[
    ["cleaned_first_name", "cleaned_last_name", "first_nickname", "last_nickname"]
] = df_first_last.apply(process_names, axis=1)
# _LOG.info(
#     "Separate transformation =\n %s",
#     hpandas.df_to_str(df_first_last, log_level=logging.WARNING),
# )


df_first_last["is_modified"] = (df_first_last["first_name"] != df_first_last["cleaned_first_name"]) | (df_first_last["last_name"] != df_first_last["cleaned_last_name"])

col_names = "origin first_name last_name cleaned_first_name cleaned_last_name first_nickname last_nickname is_modified".split()
df_first_last[col_names].head()

,origin,first_name,last_name,cleaned_first_name,cleaned_last_name,first_nickname,last_nickname,is_modified
hash,,,,,,,,
4a4079215e5eccf89e6089f7dd6574f7,Folkapp,David,Rossow,David,Rossow,,,False
bf436a1cde88d1e0079a889081d32f88,Folkapp,Jeff,Ehlers,Jeff,Ehlers,,,False
b95b4cb1a59603c67b1e1f6b4f29c81b,Folkapp,Jeremy,Schneider,Jeremy,Schneider,,,False
45f4d41f6ddf5919c35f0fecf614fa6c,Folkapp,Michael,Kam,Michael,Kam,,,False
aabc181457c8785fd6e1858a1ad7d1cd,Folkapp,Pierre,Omidyar,Pierre,Omidyar,,,False


In [42]:
df_tmp = df_first_last
hpandas.filter_df(df_tmp, "is_modified", True)[col_names]

INFO  selected=165 / 607 = 27.18%


,origin,first_name,last_name,cleaned_first_name,cleaned_last_name,first_nickname,last_nickname,is_modified
hash,,,,,,,,
38d81474c9c9563df1ccb47d92ccac1a,GP_LIn_connections,Todd,"Bendell, CFP®",Todd,Bendell,,,True
e96f296f531b6919da878ee1cd0b08a7,hedge_fund_list,Nancy A.,Kukacka,Nancy,Kukacka,,,True
cdf15b6eea41b8392a3532ccbf2cc147,GP_LIn_connections,Francis X.,Frecentese,Francis,Frecentese,,,True
a1336845ba79bfc95c3ed4461df5795e,hedge_fund_list,Thomas E.,Claugus,Thomas,Claugus,,,True
5e193c3c7b1dd661d1e48cfb3d505d7c,GP_LIn_connections,Stephanie,"Rumold, CFP®",Stephanie,Rumold,,,True
e6f35e35d6d684289e48d05a4a174bd1,hedge_fund_list,John D.,Gottfurcht,John,Gottfurcht,,,True
dd9f78e9dc5dcf9278e79701c76a3af6,GP_LIn_connections,Kristin,"Milchanowski, Ph.D.",Kristin,Milchanowski,,,True
60b1271693297e93da9b7805512099f6,GP_LIn_connections,Jose M.,"Plehn, Ph.D.",Jose,Plehn,,,True
52f76202800ca6ee141f5090e80feee2,hedge_fund_list,John P.,Calamos,John,Calamos,,,True


In [43]:
hyamm.save_to_gsheet(df_first_last, name="financial2")

https://docs.google.com/spreadsheets/d/1W8lWkaoSyX5uODoJtjiujKjijWD_I1ZrvXx7e_H5j3Y
INFO  Saved to financial2
